In [38]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import gurobipy as gp
from gurobipy import GRB

print("Entorno cargado correctamente")
print("Gurobi:", gp.gurobi.version())

Entorno cargado correctamente
Gurobi: (13, 0, 3)


In [39]:
# Elegimos la instancia que queremos resolver
archivo = Path("DATOS/INST_MX_14_PRODUCTOS_18_ZONAS.xlsx")

# Verificamos que Python encuentre el archivo
print("Archivo:", archivo)
print("¿Existe?:", archivo.exists())

# Revisamos las hojas disponibles
xls = pd.ExcelFile(archivo)
print("Hojas:", xls.sheet_names)

Archivo: DATOS/INST_MX_14_PRODUCTOS_18_ZONAS.xlsx
¿Existe?: True
Hojas: ['Contenedor', 'Productos', 'Zonas', 'VALIDACION', 'DETERIORO_TERMICO', 'EMPAQUES', 'FUENTES']


In [40]:
# ============================================================
# CARGA DE DATOS DE LA INSTANCIA
# ============================================================

contenedor = pd.read_excel(archivo, sheet_name="Contenedor")
productos = pd.read_excel(archivo, sheet_name="Productos")
zonas = pd.read_excel(archivo, sheet_name="Zonas")

# Limpiamos posibles espacios en los nombres de las columnas
contenedor.columns = contenedor.columns.str.strip()
productos.columns = productos.columns.str.strip()
zonas.columns = zonas.columns.str.strip()

# Mostramos los datos cargados
print("=== CONTENEDOR ===")
display(contenedor)

print("\n=== PRODUCTOS ===")
display(productos)

print("\n=== ZONAS ===")
display(zonas)

=== CONTENEDOR ===


,Conjunto,Valor
0,Z,18
1,H,1
2,A,14
3,L,6



=== PRODUCTOS ===


,Producto,n,w,s,alpha
0,Lechuga romana (LR),160,10,27,10
1,Lechuga iceberg (LI),146,11,29,10
2,Brócoli (B),140,12,34,9
3,Coliflor (COL),130,12,35,8
4,Zanahoria (ZAN),126,18,62,4
5,Repollo verde (RV),120,16,58,5
6,Apio (AP),112,12,33,9
7,Espinaca (ESP),103,8,22,10
8,Acelga (ACE),92,9,24,9
9,Rábano (RAD),82,10,29,5



=== ZONAS ===


,Zona,theta
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
5,6,0
6,7,1
7,8,1
8,9,1
9,10,1


In [41]:
# ============================================================
# CONJUNTOS Y PARÁMETROS DEL MODELO
# ============================================================

# Dimensiones del contenedor
param = dict(zip(contenedor["Conjunto"], contenedor["Valor"]))

Z = range(1, int(param["Z"]) + 1)
H = range(1, int(param["H"]) + 1)
A = range(1, int(param["A"]) + 1)
L = range(1, int(param["L"]) + 1)

# Conjunto de productos
P = productos["Producto"].tolist()

# Parámetros de los productos
n = dict(zip(productos["Producto"], productos["n"]))
w = dict(zip(productos["Producto"], productos["w"]))
s = dict(zip(productos["Producto"], productos["s"]))
alpha = dict(zip(productos["Producto"], productos["alpha"]))

# Parámetro de las zonas térmicas
theta = dict(zip(zonas["Zona"], zonas["theta"]))

# Penalización térmica producto-zona
d = {
    (p, z): alpha[p] * theta[z]
    for p in P
    for z in Z
}

# ============================================================
# RESUMEN DE LA INSTANCIA
# ============================================================

capacidad = len(Z) * len(H) * len(A) * len(L)
demanda = sum(n.values())
ocupacion = 100 * demanda / capacidad

print("=== RESUMEN DE LA INSTANCIA ===")
print(f"Productos:              {len(P)}")
print(f"Zonas (Z):              {len(Z)}")
print(f"Hileras (H):            {len(H)}")
print(f"Posiciones ancho (A):   {len(A)}")
print(f"Niveles (L):            {len(L)}")
print(f"Capacidad física:       {capacidad} cajas")
print(f"Demanda total:          {demanda} cajas")
print(f"Ocupación:              {ocupacion:.1f}%")

# Tamaño potencial de x[p,z,h,a,l]
num_x = len(P) * capacidad
print(f"Variables binarias x:   {num_x:,}")

=== RESUMEN DE LA INSTANCIA ===
Productos:              14
Zonas (Z):              18
Hileras (H):            1
Posiciones ancho (A):   14
Niveles (L):            6
Capacidad física:       1512 cajas
Demanda total:          1450 cajas
Ocupación:              95.9%
Variables binarias x:   21,168


In [42]:
# ============================================================
# MODELO DE OPTIMIZACIÓN
# ============================================================

m = gp.Model("Estiba_Refrigerada")

# Variable binaria:
# x[p,z,h,a,l] = 1 si el producto p ocupa esa posición
#                 0 en caso contrario
x = m.addVars(
    P, Z, H, A, L,
    vtype=GRB.BINARY,
    name="x"
)

m.update()

print("Modelo creado correctamente")
print(f"Variables creadas: {m.NumVars:,}")

Modelo creado correctamente
Variables creadas: 21,168


In [43]:
# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================

m.setObjective(
    gp.quicksum(
        d[p, z] * x[p, z, h, a, l]
        for p in P
        for z in Z
        for h in H
        for a in A
        for l in L
    ),
    GRB.MINIMIZE
)

print("Función objetivo definida correctamente")

Función objetivo definida correctamente


In [44]:
# ============================================================
# RESTRICCIÓN 1: DEMANDA
# ============================================================

for p in P:
    m.addConstr(
        gp.quicksum(
            x[p, z, h, a, l]
            for z in Z
            for h in H
            for a in A
            for l in L
        ) == n[p],
        name=f"demanda_{p}"
    )

m.update()

print("Restricciones de demanda agregadas correctamente")
print(f"Restricciones actuales: {m.NumConstrs}")

Restricciones de demanda agregadas correctamente
Restricciones actuales: 14


In [45]:
# ============================================================
# RESTRICCIÓN (2)
# Número de cajas de cada producto
# ============================================================

##### la función .addConstr() agrega la restricción al modelo
### En este caso, el modelo se define como m

for p in P:

    m.addConstr(

        gp.quicksum(
            x[p,z,h,a,l]
            for z in Z
            for h in H
            for a in A
            for l in L
        ) == n[p],

        name=f"Demanda_{p}" ###NOMBRE DE LA EC.

    )


In [46]:
# ============================================================
# RESTRICCIÓN 3: OCUPACIÓN
# Máximo una caja por posición física
# ============================================================

for z in Z:
    for h in H:
        for a in A:
            for l in L:
                m.addConstr(
                    gp.quicksum(
                        x[p, z, h, a, l]
                        for p in P
                    ) <= 1,
                    name=f"ocupacion_{z}_{h}_{a}_{l}"
                )

m.update()

print("Restricciones de ocupación agregadas correctamente")
print(f"Restricciones actuales: {m.NumConstrs}")

Restricciones de ocupación agregadas correctamente
Restricciones actuales: 1540


In [47]:
# ============================================================
# RESTRICCIÓN (4)
# No puede haber cajas flotando
# ============================================================

for z in Z:
    for h in H:
        for a in A:
            # l > 1
            for l in L[1:]:

                m.addConstr(

                    gp.quicksum(
                        x[p,z,h,a,l]
                        for p in P
                    )

                    <=

                    gp.quicksum(
                        x[p,z,h,a,l-1]
                        for p in P
                    ),

                    name=f"Posicion_{z}_{a}_{l}"
                )

m.update()

print("Restricciones de no flotación agregadas")
print(f"Restricciones actuales: {m.NumConstrs}")

Restricciones de no flotación agregadas
Restricciones actuales: 2800


In [48]:
# ============================================================
# RESTRICCIÓN (5)
# Peso máximo soportado
# ============================================================

for z in Z:
    for h in H:
        for a in A:

            # l < |L_z|
            for l in list(L)[:-1]:

                peso_superior = gp.quicksum(

                    w[p] * x[p,z,h,a,ll]

                    for p in P
                    for ll in L
                    if ll > l
                )

                resistencia = gp.quicksum(

                    s[p] * x[p,z,h,a,l]

                    for p in P
                )

                m.addConstr(

                    peso_superior <= resistencia,

                    name=f"Apilamiento_{z}_{h}_{a}_{l}"
                )

m.update()

print("Restricciones de apilamiento agregadas")
print(f"Total de variables: {m.NumVars}")
print(f"Total de restricciones: {m.NumConstrs}")

Restricciones de apilamiento agregadas
Total de variables: 21168
Total de restricciones: 4060


In [49]:
import time
import threading

inicio = None
resolver_terminado = False

def mostrar_timer():
    while not resolver_terminado:
        transcurrido = time.time() - inicio

        horas = int(transcurrido // 3600)
        minutos = int((transcurrido % 3600) // 60)
        segundos = int(transcurrido % 60)

        print(
            f"\r⏱ Tiempo transcurrido: "
            f"{horas:02d}:{minutos:02d}:{segundos:02d}",
            end="",
            flush=True
        )

        time.sleep(1)

In [50]:


# ============================================================
# OPTIMIZAR + CRONÓMETRO
# ============================================================

inicio = time.time()
resolver_terminado = False

# Iniciar cronómetro en paralelo
hilo_timer = threading.Thread(target=mostrar_timer)
hilo_timer.start()

try:
    m.optimize()

finally:
    resolver_terminado = True
    hilo_timer.join()

tiempo_total = time.time() - inicio

horas = int(tiempo_total // 3600)
minutos = int((tiempo_total % 3600) // 60)
segundos = tiempo_total % 60

print(
    f"\n\n✅ Optimización terminada en "
    f"{horas:02d}:{minutos:02d}:{segundos:05.2f}"
)

# ============================================================
# INFORMACIÓN DE LA RESOLUCIÓN
# ============================================================

print("\n" + "="*60)
print("RESUMEN DE LA RESOLUCIÓN")
print("="*60)
print(f"Total de variables:      {m.NumVars:,}")
print(f"Total de restricciones:  {m.NumConstrs:,}")
print(f"Tiempo Gurobi:           {m.Runtime:.4f} segundos")
print(f"Estado del modelo:       {m.Status}")
print("="*60)

⏱ Tiempo transcurrido: 00:00:00Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[x86] - Darwin 22.6.0 22H730)

CPU model: Intel(R) Core(TM) i5-7360U CPU @ 2.30GHz
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 4060 rows, 21168 columns and 169344 nonzeros (Min)
Model fingerprint: 0xa93f0607
Model has 14112 linear objective coefficients
Variable types: 0 continuous, 21168 integer (21168 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [4e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+02]

⏱ Tiempo transcurrido: 00:00:01Presolve removed 14 rows and 0 columns
Presolve time: 1.51s
Presolved: 4046 rows, 21168 columns, 148176 nonzeros
Variable types: 0 continuous, 21168 integer (21168 binary)
⏱ Tiempo transcurrido: 00:00:02
Use crossover to convert LP symmetric solution to basic solution...
Crossover time: 0.09 seconds (0.02 work units)

Root relaxation: objective 9.39290

In [51]:
# ============================================================
# ESTADÍSTICAS DEL BENCHMARK
# ============================================================

print("\n" + "="*65)
print("ESTADÍSTICAS DEL BENCHMARK")
print("="*65)

print(f"Productos:                 {len(P):,}")
print(f"Capacidad física:          {capacidad:,} cajas")
print(f"Demanda total:             {demanda:,} cajas")
print(f"Ocupación:                 {ocupacion:.2f}%")

print("-"*65)

print(f"Variables totales:         {m.NumVars:,}")
print(f"Restricciones:             {m.NumConstrs:,}")
print(f"Coeficientes no cero:      {m.NumNZs:,}")

print("-"*65)

print(f"Tiempo Gurobi:             {m.Runtime:.4f} s")
print(f"Tiempo real:               {tiempo_total:.4f} s")
print(f"Nodos explorados:          {m.NodeCount:,.0f}")
print(f"Estado Gurobi:             {m.Status}")

# Estas estadísticas requieren que Gurobi haya encontrado
# al menos una solución factible
if m.SolCount > 0:

    print(f"Soluciones encontradas:    {m.SolCount}")
    print(f"Valor objetivo:            {m.ObjVal:,.6f}")
    print(f"Mejor cota (ObjBound):     {m.ObjBound:,.6f}")
    print(f"MIP Gap:                   {m.MIPGap * 100:.6f}%")

else:

    print("Soluciones encontradas:    0")
    print("Todavía no existe una solución factible.")

print("="*65)


ESTADÍSTICAS DEL BENCHMARK
Productos:                 14
Capacidad física:          1,512 cajas
Demanda total:             1,450 cajas
Ocupación:                 95.90%
-----------------------------------------------------------------
Variables totales:         21,168
Restricciones:             4,060
Coeficientes no cero:      169,344
-----------------------------------------------------------------
Tiempo Gurobi:             10529.2939 s
Tiempo real:               10529.6008 s
Nodos explorados:          16,245
Estado Gurobi:             2
Soluciones encontradas:    10
Valor objetivo:            9,955.000000
Mejor cota (ObjBound):     9,955.000000
MIP Gap:                   0.000000%


In [52]:
# ============================================================
# RESULTADOS
# ============================================================

if m.status == GRB.OPTIMAL:

    print("\n")
    print("="*60)
    print("VALOR ÓPTIMO")
    print("="*60)
    print(m.objVal)

    for z in Z:

        print("\n")
        print("="*40)
        print(f"ZONA {z}")
        print("="*40)

        # Hilera dentro de la zona
        for h in H:

            print(f"\n----- Hilera {h} -----")

            # Posiciones dentro de la hilera
            for a in A:

                print(f"\nPosición {a}")

                # Niveles de arriba hacia abajo
                for l in reversed(L):

                    producto = "-"

                    for p in P:

                        if x[p,z,h,a,l].X > 0.5:
                            producto = p
                            break

                    print(f"Nivel {l}: {producto}")

else:

    print("No se encontró solución óptima.")



VALOR ÓPTIMO
9955.0


ZONA 1

----- Hilera 1 -----

Posición 1
Nivel 6: Perejil (PER)
Nivel 5: Perejil (PER)
Nivel 4: Espinaca (ESP)
Nivel 3: Cilantro (CIL)
Nivel 2: Lechuga romana (LR)
Nivel 1: Brócoli (B)

Posición 2
Nivel 6: Espinaca (ESP)
Nivel 5: Espinaca (ESP)
Nivel 4: Espinaca (ESP)
Nivel 3: Lechuga romana (LR)
Nivel 2: Brócoli (B)
Nivel 1: Repollo verde (RV)

Posición 3
Nivel 6: Cilantro (CIL)
Nivel 5: Lechuga romana (LR)
Nivel 4: Espinaca (ESP)
Nivel 3: Lechuga romana (LR)
Nivel 2: Brócoli (B)
Nivel 1: Repollo verde (RV)

Posición 4
Nivel 6: Espárrago (ESPA)
Nivel 5: Perejil (PER)
Nivel 4: Perejil (PER)
Nivel 3: Espárrago (ESPA)
Nivel 2: Lechuga romana (LR)
Nivel 1: Brócoli (B)

Posición 5
Nivel 6: Lechuga romana (LR)
Nivel 5: Cilantro (CIL)
Nivel 4: Espinaca (ESP)
Nivel 3: Lechuga romana (LR)
Nivel 2: Brócoli (B)
Nivel 1: Repollo verde (RV)

Posición 6
Nivel 6: Perejil (PER)
Nivel 5: Cilantro (CIL)
Nivel 4: Perejil (PER)
Nivel 3: Espinaca (ESP)
Nivel 2: Lechuga romana (LR)


In [53]:
#   VISUALICAZION


def agregar_caja(fig, x, y, z, dx, dy, dz, color, texto, hover):

    vertices = np.array([
        [x, y, z],
        [x+dx, y, z],
        [x+dx, y+dy, z],
        [x, y+dy, z],
        [x, y, z+dz],
        [x+dx, y, z+dz],
        [x+dx, y+dy, z+dz],
        [x, y+dy, z+dz]
    ])

    I=[0,0,0,1,1,2,4,4,5,6,2,3]
    J=[1,2,3,2,5,3,5,7,6,7,6,7]
    K=[2,3,1,5,6,7,7,6,4,4,1,0]

    fig.add_trace(

        go.Mesh3d(

            x=vertices[:,0],
            y=vertices[:,1],
            z=vertices[:,2],

            i=I,
            j=J,
            k=K,

            color=color,
            opacity=0.95,
            flatshading=True,

            hovertext=hover,
            hoverinfo="text",
            showscale=False

        )

    )

    fig.add_trace(

        go.Scatter3d(

            x=[x+dx/2],
            y=[y+dy/2],
            z=[z+dz/2],

            mode="text",

            text=[texto],

            textfont=dict(
                color="white",
                size=12
            ),

            showlegend=False,

            hoverinfo="skip"

        )

    )

In [95]:
color_producto = {
    "Brócoli (B)": "springgreen",
    "Lechuga (LE)": "lime",
    "Zanahoria (ZAN)": "orange",
    "Jitomate (J)": "lightcoral",
    "Pepino (PE)": "darkgreen",
    "Coliflor (COL)": "lightgray",
    "Chile morrón (CHI)": "mediumaquamarine",
    "Calabaza CA)": "olive",
    "Fresa (F)": "red",
    "Papa (P)": "saddlebrown",
    "Manzana (M)": "gold",
    "Cebolla (C)": "purple",
    "Aguacate (A)": "darkslategrey",
    "Limón (LI)": "yellow",
    "Zarzamora (ZA)": "mediumslateblue",
    "Frambuesa (FR)": "hotpink",
    "Repollo (REP)": "lightgreen",
    "Apio (AP)": "darkolivegreen",
    "Lechuga romana (LR)": "darkseagreen",
    "Lechuga iceberg (LI)": "lightblue",
    "Repollo verde (RV)": "darkgreen",
    "Espinaca (ESP)": "yellowgreen",
    "Acelga (ACE)": "mediumseagreen",
    "Rábano (RAD)": "crimson",
    "Betabel (BET)": "darkred",
    "Espárrago (ESPA)": "lightseagreen",
    "Perejil (PER)": "darkkhaki",
    "Cilantro (CIL)": "yellow"
}

In [96]:
# Crear la figura 3D
fig = go.Figure()

In [97]:
espacio_zona = len(A)+1

for p in P:

    for z in Z:

        for h in H:

            for a in A:

                for l in L:

                    if x[p,z,h,a,l].X > 0.5:

                        X = (z-1)*espacio_zona
                        Y = a-1
                        Zplot = l-1

                        hover = (
                            f"<b>Producto:</b> {p}<br>"
                            f"<b>Zona:</b> {z}<br>"
                            f"<b>Hilera:</b> {h}<br>"
                            f"<b>Posición:</b> {a}<br>"
                            f"<b>Nivel:</b> {l}<br>"
                            f"<b>Peso:</b> {w[p]} kg<br>"
                            f"<b>Resistencia:</b> {s[p]} kg<br>"
                            f"<b>Temperatura:</b> {theta[z]} °C"
                        )

                        agregar_caja(

                            fig,

                            X,
                            Y,
                            Zplot,

                            0.9,
                            0.9,
                            0.9,

                            color_producto[p],

                            p,

                            hover

                        )

In [98]:
for z in Z:

    X = (z-1)*espacio_zona

    vertices = np.array([
        [X,0,0],
        [X+1,0,0],
        [X+1,len(A),0],
        [X,len(A),0],
        [X,0,len(L)],
        [X+1,0,len(L)],
        [X+1,len(A),len(L)],
        [X,len(A),len(L)]
    ])

    fig.add_trace(

        go.Mesh3d(

            x=vertices[:,0],
            y=vertices[:,1],
            z=vertices[:,2],

            i=[0,0,0,1,1,2,4,4,5,6,2,3],
            j=[1,2,3,2,5,3,5,7,6,7,6,7],
            k=[2,3,1,5,6,7,7,6,4,4,1,0],

            color="gray",

            opacity=0.08,

            hoverinfo="skip",

            showscale=False

        )

    )
    

In [99]:
fig.update_layout(

    title="Distribución óptima de cajas",

    scene=dict(

        xaxis=dict(
            title="Zona",
            tickvals=[(z-1)*espacio_zona for z in Z],
            ticktext=[f"Z{z}" for z in Z]
        ),

        yaxis=dict(
            title="Posición",
            tickvals=[a-1 for a in A],
            ticktext=[f"A{a}" for a in A]
        ),

        zaxis=dict(
            title="Nivel",
            tickvals=[l-1 for l in L],
            ticktext=[f"N{l}" for l in L]
        ),

        aspectmode="data"

    ),

    width=1200,

    height=750

)

fig.show()